# EmbeddingGemma 2 - text embeddings, similarity, and semantic search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/embeddinggemma-2-inference-similarity.ipynb)

Run Google DeepMind's [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2) locally with
`sentence-transformers`: encode queries and documents with task prompts, inspect the 768-dim vectors,
rank documents by cosine similarity, and answer questions with a tiny FAQ semantic search.

EmbeddingGemma 2 is an Apache 2.0 multimodal embedding model (740M total: a 270M text model plus
vision and audio encoders) that maps text, images, video, and audio into one 768-dimensional space.
This notebook only needs text, so it skips the vision and audio encoders to save RAM.

**Runtime:** pick *Runtime > Change runtime type > T4 GPU* for the fastest run. A CPU runtime also works,
but the first load downloads the full checkpoint (~1.5 GB) even in text-only mode, and encoding is slower.

**Links:** [model card](https://ai.google.dev/gemma/docs/embeddinggemma/model_card_2) ·
[Hugging Face](https://huggingface.co/google/embeddinggemma-2) ·
[announcement](https://blog.google/innovation-and-ai/technology/developers-tools/embeddinggemma-2/) ·
[developer guide](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/) ·
[Sentence Transformers inference guide](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers)

In [1]:
%pip install -q -U "sentence-transformers>=6.1.0" "transformers>=5.19.0"
%pip install -q pillow torchvision

# Upgrade both packages (as in the model card quick start): Colab's preinstalled transformers
# already satisfies sentence-transformers' requirement, so it is not upgraded otherwise, and
# the `embedding_gemma2` architecture was first added in transformers 5.19.0.
# The model's processor imports pillow and torchvision even for text-only use;
# both are preinstalled on Colab, so the second line is a no-op there.

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported the old transformers, so the upgrade only takes effect after a restart.
> Skipping this gives `Transformers does not recognize this architecture` for `embedding_gemma2`.

In [2]:
import sentence_transformers
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.19.0"), (
    f"transformers {transformers.__version__} is loaded; EmbeddingGemma 2 needs 5.19.0+. "
    "Run the install cell, then Runtime > Restart session."
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# float16 is NOT supported: activations overflow and you get NaN or silently degraded embeddings.
# Use bfloat16 on GPUs with native support (A100, L4, ...) and float32 elsewhere (T4, CPU).
if DEVICE == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False):
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float32

print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", DEVICE, "| dtype:", DTYPE)

sentence-transformers 6.1.0
transformers 5.19.0
torch 2.14.1+cpu
device: cpu | dtype: torch.float32


## 1. Load a text-only EmbeddingGemma 2

`config_kwargs={"vision_config": None, "audio_config": None}` drops the vision (170M) and audio (300M)
encoders, leaving the 270M text model. Text embeddings are unchanged.

In [3]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "google/embeddinggemma-2"

model = SentenceTransformer(
    MODEL_ID,
    device=DEVICE,
    config_kwargs={"vision_config": None, "audio_config": None},
    model_kwargs={"torch_dtype": DTYPE},
)

print("parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print("dtype:", model.dtype)
print("embedding dim:", model.get_embedding_dimension())
print("similarity fn:", model.similarity_fn_name)

parameters: 271M
dtype: torch.float32
embedding dim: 768
similarity fn: cosine


## 2. Encode with task prompts

EmbeddingGemma 2 is trained with short task prefixes. For search, encode queries with
`prompt_name="SearchQuery"` and corpus texts with `prompt_name="Document"`. For symmetric tasks use the
same prompt on both sides (`SentenceSimilarity`, `Classification`, `Clustering`).

In [4]:
for name in ["SearchQuery", "Document", "QuestionAnswering", "SentenceSimilarity", "Classification", "Clustering"]:
    print(f"{name:>18}: {model.prompts[name]!r}")

       SearchQuery: 'task: search result | query: '
          Document: 'title: none | text: '
 QuestionAnswering: 'task: question answering | query: '
SentenceSimilarity: 'task: sentence similarity | query: '
    Classification: 'task: classification | query: '
        Clustering: 'task: clustering | query: '


In [5]:
query = "What causes the northern lights?"
document = "The northern lights are caused by charged particles from the sun."

query_emb = model.encode(query, prompt_name="SearchQuery")
doc_emb = model.encode(document, prompt_name="Document")

print("query embedding shape:", query_emb.shape)
print("first 5 values:", query_emb[:5].round(4))
print("L2 norm:", round(float((query_emb**2).sum() ** 0.5), 4))
print("similarity:", model.similarity(query_emb, doc_emb))

query embedding shape: (768,)
first 5 values: [ 0.0125 -0.0028  0.0402 -0.0183 -0.0406]
L2 norm: 1.0
similarity: tensor([[0.8707]])


## 3. Rank documents by cosine similarity

The model normalizes its outputs, so `model.similarity` (cosine) ranks every document against the query.
Scores skew high (unrelated text often lands around 0.5-0.7), so compare them relatively.

In [6]:
query = "How do I keep my sourdough starter healthy?"
documents = [
    "Feed your starter equal weights of flour and water every day and keep it at room temperature.",
    "A sourdough loaf needs a long, slow proof to develop its sour flavor.",
    "Tomato plants need at least six hours of direct sunlight a day.",
    "Stock prices fell sharply after the quarterly earnings report.",
    "Discard half of the starter before each feeding to keep the yeast active.",
]

query_emb = model.encode(query, prompt_name="SearchQuery")
doc_embs = model.encode(documents, prompt_name="Document")
scores = model.similarity(query_emb, doc_embs)[0]

print("query:", query)
for rank, idx in enumerate(scores.argsort(descending=True).tolist(), start=1):
    print(f"{rank}. {scores[idx]:.4f}  {documents[idx]}")

query: How do I keep my sourdough starter healthy?
1. 0.8032  Feed your starter equal weights of flour and water every day and keep it at room temperature.
2. 0.7814  Discard half of the starter before each feeding to keep the yeast active.
3. 0.7756  A sourdough loaf needs a long, slow proof to develop its sour flavor.
4. 0.5916  Tomato plants need at least six hours of direct sunlight a day.
5. 0.5170  Stock prices fell sharply after the quarterly earnings report.


## 4. Semantic search over a small FAQ

Embed the FAQ once, then match each user question to the closest entry. When a document has a real
title, format it as `title: {title} | text: {content}` (here the FAQ question is the title);
`prompt_name="Document"` is the shortcut for `title: none | text: `.

In [7]:
faq = {
    "How do I request a refund?": "If you were charged by mistake or more than once, request a refund from Billing > Payments. Refunds arrive within 5 business days.",
    "I never received my verification code": "Check your spam folder or resend the code after 60 seconds.",
    "Can I change my plan?": "Upgrade or downgrade any time from Billing settings.",
    "How do I delete my account?": "Go to Settings > Privacy and choose Delete account.",
    "Do you ship internationally?": "We ship to over 40 countries; rates appear at checkout.",
}

faq_texts = [f"title: {question} | text: {answer}" for question, answer in faq.items()]
faq_embs = model.encode(faq_texts)

user_questions = [
    "I got billed twice, can I get my money back?",
    "The login code email never showed up",
    "Can you send my order to Japan?",
    "I want to close my account for good",
    "How do I switch to the annual plan?",
]
question_embs = model.encode(user_questions, prompt_name="SearchQuery")
scores = model.similarity(question_embs, faq_embs)

faq_items = list(faq.items())
for question, row in zip(user_questions, scores):
    best = int(row.argmax())
    title, answer = faq_items[best]
    print(f"query: {question}\n  best match ({row[best]:.4f}): {title} -> {answer}\n")

query: I got billed twice, can I get my money back?
  best match (0.7716): How do I request a refund? -> If you were charged by mistake or more than once, request a refund from Billing > Payments. Refunds arrive within 5 business days.

query: The login code email never showed up
  best match (0.7679): I never received my verification code -> Check your spam folder or resend the code after 60 seconds.

query: Can you send my order to Japan?
  best match (0.6756): Do you ship internationally? -> We ship to over 40 countries; rates appear at checkout.

query: I want to close my account for good
  best match (0.7901): How do I delete my account? -> Go to Settings > Privacy and choose Delete account.

query: How do I switch to the annual plan?
  best match (0.7425): Can I change my plan? -> Upgrade or downgrade any time from Billing settings.



## Next steps

- **Smaller vectors:** EmbeddingGemma 2 supports Matryoshka truncation to 512, 256, or 128 dims with
  `model.encode(..., truncate_dim=256, normalize_embeddings=True)`; queries and documents must use the same size.
- **Other tasks:** try `prompt_name="Classification"` for sentiment-style labels or `"Clustering"` for custom categories.
- **Images, video, audio:** load without `config_kwargs` (full 740M model) and see the
  [developer guide](https://developers.googleblog.com/embeddinggemma-2-the-developer-guide/).